# **Hill Country Grocer Weekly Revenue Prediction**: Supervised Tabular Regression with Six-Model Ensemble
---

In order to help a Texas regional grocery chain forecast weekly revenue per item-store combination from product and store attributes alone, a six-model regression ensemble was developed across 8,880 weekly product-store sales records spanning multiple banners, regions, and departments. The target is `Weekly_Revenue_USD`. The operator's target performance band on this dataset is an R² of approximately **0.85-0.95** and a Mean Absolute Percentage Error in the **8-15%** range on a held-out test set; actual measured values are filled in after the notebook is executed in Colab. Practically, this means a category manager planning shelf allocation, pricing, or promotion strategy for a specific item at a specific store can plug the product attributes (department, brand type, weight, pack size, list price, promo price, shelf facings) and store attributes (banner, region, square footage, age) into a deployed app and receive a weekly revenue forecast rooted in measured historical patterns across the chain.

The fitted model is selected from a six-model bake-off (Decision Tree, Bagging, Random Forest, Gradient Boosting, XGBoost, CatBoost) ranked by repeated K-fold cross-validation RMSE and confirmed on a fully held-out test split. Critically, `Weekly_Units_Sold` is dropped from the feature set before training: units sold multiplied by price is effectively revenue, making it a target-leakage feature that trivializes the prediction problem and is useless in deployment (a revenue forecaster that requires knowing units sold to predict revenue is not a forecaster). All six trained candidates are serialized to disk to support a downstream frontend that exposes a model-choice dropdown across the full bake-off, not just the primary.

## **Executive Summary**

### **Business Opportunities**

**A. Forecast weekly revenue per item-store combination consistently across the chain.** A multi-banner regional grocer with thousands of item-store combinations cannot price, stock, or merchandise each one by intuition. The fitted model gives every category manager and inventory planner the same defensible reference forecast for any combination in the catalog, rooted in measured patterns across 8,880 historical weekly records.

**B. Plan shelf allocation and promotional pricing with measured confidence intervals.** When a store needs to decide how many shelf facings to allocate or whether to run a promotional price on a specific item, the fitted model returns a revenue estimate in seconds. The MAPE figure (filled in after execution) gives planners a measured confidence band: forecasts are typically off by that percentage of actual on any given item-store row.

**C. Identify mispriced, misallocated, or under-promoted inventory.** Comparing the model's revenue forecast against actual weekly sales per item-store combination surfaces rows where the actual materially diverges from the forecast — overstocked (forecast materially exceeds actuals) or underpromoted (actuals materially exceed forecast). The discrepancy becomes a sortable list of action items for category review.

### **Outcomes**

_Observations: to be populated after execution._ The primary model's test-set MAE, MAPE, R², and cross-validation stability are reported in the Test Set Evaluation and Business Alignment sections below. Per the catalog's quality-tier framework (`system-design.md` §6.1), this build's tier assignment is determined by the measured test R² and MAPE against the v1.0 thresholds (Tier 1: R² ≥ 0.92 and MAPE ≤ 8%; Tier 2: R² ≥ 0.85 and MAPE ≤ 12%; Tier 3: R² ≥ 0.70 and MAPE ≤ 20%; Tier 4 otherwise).

---

## **Problem Statement**

A Texas regional grocery chain faces profit leakage driven by weekly demand variability at the item-store level, where even modest forecast errors translate into excess inventory, lost sales, and margin compression across thousands of item-store combinations. This model predicts weekly revenue for each item-store combination from product and store attributes alone (no historical units-sold input), to support shelf allocation, pricing, and promotion decisions.

**Business context.** Small weekly forecasting errors scale into material financial impact across the chain's product catalog and store footprint. Overstock ties up working capital and increases holding costs. Understock directly reduces revenue capture. Improving prediction accuracy compounds into measurable gains in margin stability and inventory efficiency.

**Objective.** Build and evaluate regression models to forecast weekly item-store revenue from product and store attributes, select the model with the strongest out-of-sample performance, and establish a deployable forecasting capability to support planning decisions.

## **Data**

The dataset contains 8,880 weekly item-store revenue records from a synthetic Hill Country Grocer panel (Texas regional grocery chain). Source: synthetic dataset generated for this reference build, committed to `000-smb-consulting-reference-data` per the system's data-loading contract.

| Column | Description |
| --- | --- |
| `UPC` | 12-digit product identifier (high cardinality; dropped before modeling) |
| `Item_Description` | Product name / description |
| `Department` | Department / aisle category |
| `Brand_Type` | Brand classification (e.g., national brand, private label) |
| `Net_Weight_oz` | Product weight in ounces |
| `Pack_Size` | Number of units per pack |
| `List_Price_USD` | Full retail price per unit in USD |
| `Promo_Price_USD` | Promotional price per unit in USD (less than or equal to list) |
| `Shelf_Facings` | Number of shelf facings allocated to the item at the store |
| `Store_Number` | Store identifier |
| `Store_Banner` | Banner name (chain or sub-brand the store operates under) |
| `Store_Region` | Geographic region within Texas |
| `Store_Sq_Ft` | Store square footage |
| `Store_Open_Year` | Year the store opened (transformed to `store_age` for modeling) |
| `Weekly_Units_Sold` | Units sold that week (**dropped before modeling — target leakage**; see callout cell below) |
| `Weekly_Revenue_USD` | **Target.** Weekly revenue (USD) for that item at that store |

---

# **Code Execution**

### **Runtime Configuration**

> **Hardware Accelerator:** **CPU** (this notebook trains tree-based ensembles which are CPU-bound; GPU is not required)
>
> **RAM:** **Standard** (the dataset at 8,880 rows fits in default Colab memory comfortably)
>
> Please ensure the Colab runtime is set to this configuration before executing any code cells. The model training step (Model Training Loop) runs six models with `GridSearchCV` hyperparameter tuning and repeated K-fold cross-validation; expect that step to take **8-15 minutes** on standard Colab CPU runtime.

### **Library Installation**

**Summary:** Required packages were checked against the active environment and missing ones installed. A runtime-restart banner is shown if any package was installed, so the imports cell below picks up the freshly installed versions.

**Observations:** All dependencies are tree-based regression libraries (`xgboost`, `catboost`) plus the standard PyData stack (`scikit-learn`, `pandas`, `numpy`, `matplotlib`, `seaborn`, `joblib`). The Colab base image already includes most of these; `catboost` is typically the only fresh install.

In [ ]:
# ------------------------------
# LIBRARY INSTALLATION
# ------------------------------
# Check for required packages and install any that are missing.
# If anything was installed, display a styled banner directing the user
# to restart the Colab runtime so the imports cell below picks up the
# newly installed versions.

import importlib                                    # used to probe whether a package is importable without importing it for real
import subprocess                                   # used to invoke pip from within the notebook process
import sys                                          # provides sys.executable so pip targets the active interpreter
from IPython.display import HTML, display           # used to render the styled restart banner

REQUIRED_PACKAGES = [                               # one entry per pip-installable name; import name in comment when it differs
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "scikit-learn",                                 # import name is "sklearn"
    "joblib",
    "xgboost",
    "catboost",
]

IMPORT_NAME_OVERRIDES = {                           # pip-name to import-name mapping where they differ
    "scikit-learn": "sklearn",
}

installed_any = False                               # tracks whether the restart banner should render

for pkg in REQUIRED_PACKAGES:
    import_name = IMPORT_NAME_OVERRIDES.get(pkg, pkg)
    try:
        importlib.import_module(import_name)        # cheap check: can we import the package at its canonical name
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pkg])
        installed_any = True

if installed_any:
    display(HTML("""
    <div style="border:2px solid #c0392b; padding:12px; border-radius:6px;
                background:#fdecea; color:#922b21; font-weight:bold;">
        Runtime Restart Required: one or more packages were installed.
        Please restart the Colab runtime (Runtime &rarr; Restart runtime)
        before running the cells below, otherwise imports may pick up
        stale package metadata.
    </div>
    """))
else:
    print("All required packages are already installed. No restart needed.")

### **Imports and Configuration**

**Summary:** Core libraries for data handling, preprocessing, modeling, evaluation, and visualization were imported. Reproducibility seeds and cross-validation configuration were pinned to fixed values used throughout the notebook.

**Observations:** The notebook environment is now ready for data loading and analysis. Random-state seeds are pinned across train/test split, `GridSearchCV`, repeated K-fold, and each individual estimator's RNG so the model bake-off is reproducible across runs.

In [ ]:
# ------------------------------
# IMPORTS AND CONFIGURATION
# ------------------------------
# Bring in the libraries used end-to-end. Comments explain each one's role.
# Pin random-state seeds and cross-validation knobs so the bake-off is
# reproducible across runs.

import os                                                 # filesystem paths for serialized model outputs
import json                                               # writing model_registry.json
import warnings                                           # suppress informational deprecation chatter during training
warnings.filterwarnings("ignore")                         # quiet pandas/sklearn deprecation warnings

import numpy as np                                        # numerical operations, reproducible RNG
import pandas as pd                                       # tabular data loading, cleaning, analysis
import matplotlib.pyplot as plt                           # charts and model-performance visualizations
import seaborn as sns                                     # statistical visualizations and plot styling
import joblib                                             # serialize fitted models / preprocessor to .joblib files
from IPython.display import display                       # explicit import so display() works under Papermill / nbconvert, not just IPython kernels

from sklearn.compose import ColumnTransformer             # apply different preprocessing to numeric vs categorical columns
from sklearn.ensemble import (                            # ensemble regressors used in the bake-off
    BaggingRegressor,
    GradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer                  # impute missing values inside the preprocessing pipeline
from sklearn.metrics import (                             # regression evaluation metrics
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import (                     # cross-validation strategies and tuning
    GridSearchCV,
    RepeatedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline                     # chain preprocessor + estimator to prevent data leakage during CV
from sklearn.preprocessing import OneHotEncoder           # convert categorical columns to model-ready indicator columns
from sklearn.tree import DecisionTreeRegressor            # low-complexity tree baseline

from xgboost import XGBRegressor                          # gradient-boosting challenger
from catboost import CatBoostRegressor                    # gradient-boosting challenger (categorical-aware, but used through OHE here)

# ---- Reproducibility and CV configuration ----
RANDOM_STATE = 1                                          # used in train/test split and individual estimator RNGs
HOLDOUT_TEST_SIZE = 0.30                                  # 30% holdout for final unbiased evaluation
SEARCH_CV = 5                                             # K-fold splits inside GridSearchCV
REPEATED_VALIDATION_CV = RepeatedKFold(                   # repeated K-fold for stable model ranking
    n_splits=5,
    n_repeats=2,
    random_state=RANDOM_STATE,
)

# ---- Display configuration ----
pd.set_option("display.max_columns", None)                # show all columns in DataFrame.head() etc.
pd.set_option("display.max_rows", 100)                    # cap row display to 100 to keep output manageable
sns.set_theme(style="whitegrid")                          # consistent seaborn theme across all plots

print(f"Random state pinned to {RANDOM_STATE}. Holdout test size = {HOLDOUT_TEST_SIZE:.0%}.")

### **Data Ingestion**

**Summary:** The Hill Country Grocer weekly sales dataset was loaded from the `000-smb-consulting-reference-data` repository via the canonical slug-pattern URL on `raw.githubusercontent.com`. This is the data-loading contract declared in `notebook-style-guide.md` v2.1: reference-build notebooks load their datasets by slug from the public reference-data repo, never from arbitrary public URLs at execution time.

**Observations:** The dataset loads as a single CSV at 8,880 rows by 16 columns. No authentication is required because the reference-data repo is public and `*.csv` is excluded from LFS in its `.gitattributes` (so `raw.githubusercontent.com` serves the actual file bytes, not an LFS pointer).

In [ ]:
# ------------------------------
# DATA INGESTION
# ------------------------------
# Load the Hill Country Grocer weekly sales CSV from the reference-data repo
# via the canonical slug-pattern URL. Note the engagement folder retains the
# older "ref-retail-revenue-pred" name for backward compatibility — the
# dataset and notebook are Hill Country Grocer.

ENGAGEMENT_FOLDER = "ref-retail-revenue-pred__reg__ensemble-exp"   # legacy folder name; do not rename
DATA_FILENAME = "hill_country_grocer_weekly_sales.csv"             # 16-column synthetic Texas grocer panel
DATA_REPO_BASE = "https://raw.githubusercontent.com/EvagAIML/000-smb-consulting-reference-data/main"
DATASET_URL = f"{DATA_REPO_BASE}/engagements/{ENGAGEMENT_FOLDER}/data/raw/{DATA_FILENAME}"

print(f"Loading dataset from: {DATASET_URL}")
df_raw = pd.read_csv(DATASET_URL)                          # read CSV directly over HTTPS into a DataFrame
print(f"Loaded shape: {df_raw.shape}")
df_raw.head()

### **Data Checkpoint**

**Summary:** A working copy of the raw DataFrame was created so subsequent cleaning and feature-engineering steps mutate the working copy rather than the originally loaded data. This makes it cheap to re-run a downstream cell after editing it without re-downloading the CSV.

**Observations:** `df_raw` is preserved untouched. All downstream operations target `data`.

In [ ]:
# ------------------------------
# DATA CHECKPOINT
# ------------------------------
# Copy the raw DataFrame so we can mutate the working copy freely without
# losing the original load.

data = df_raw.copy()                                       # working copy; df_raw remains pristine
print(f"Working DataFrame shape: {data.shape}")

### **Data Understanding**

**Summary:** Column dtypes, basic descriptive statistics, and the target's distribution shape were assessed. This pass tells us which columns are numeric vs categorical, whether any unexpected dtypes need conversion, and the rough scale and skew of `Weekly_Revenue_USD`.

**Observations:** _to be populated after execution._ Expected: six categorical columns (`Item_Description`, `Department`, `Brand_Type`, `Store_Number`, `Store_Banner`, `Store_Region`), seven numeric columns (`Net_Weight_oz`, `Pack_Size`, `List_Price_USD`, `Promo_Price_USD`, `Shelf_Facings`, `Store_Sq_Ft`, `Store_Open_Year`), plus `UPC` (identifier, to be dropped), `Weekly_Units_Sold` (target leakage, to be dropped), and the target `Weekly_Revenue_USD`.

In [ ]:
# ------------------------------
# DATA UNDERSTANDING
# ------------------------------
# Print dtype info, descriptive statistics, and a target-distribution
# summary so we can sanity-check the load before any cleaning.

print("=== dtypes ===")
print(data.dtypes)
print()

print("=== Numeric descriptive statistics ===")
display(data.describe())

print("=== Categorical descriptive statistics ===")
display(data.describe(include="object"))

print("=== Target (Weekly_Revenue_USD) distribution ===")
print(data["Weekly_Revenue_USD"].describe())
print(f"Skew:     {data['Weekly_Revenue_USD'].skew():.3f}")
print(f"Kurtosis: {data['Weekly_Revenue_USD'].kurt():.3f}")

### **Data Quality Checks**

**Summary:** The DataFrame was inspected for missing values, duplicate rows, and obviously implausible values (negative prices, zero or negative weights, promo prices above list prices). These checks fire before any feature engineering so we know what cleaning, if any, is needed.

**Observations:** _to be populated after execution._

In [ ]:
# ------------------------------
# DATA QUALITY CHECKS
# ------------------------------
# Check for missing values, duplicates, and out-of-range numerics. We do
# not yet mutate the data — these checks just report so we know what
# (if anything) the feature-engineering cell needs to repair.

print("=== Missing values per column ===")
missing = data.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values detected.")
print()

print("=== Duplicate rows ===")
print(f"{data.duplicated().sum()} duplicate rows.")
print()

print("=== Numeric range sanity checks ===")
print(f"Rows with List_Price_USD <= 0:      {(data['List_Price_USD'] <= 0).sum()}")
print(f"Rows with Promo_Price_USD <= 0:     {(data['Promo_Price_USD'] <= 0).sum()}")
print(f"Rows with Promo > List (invalid):   {(data['Promo_Price_USD'] > data['List_Price_USD']).sum()}")
print(f"Rows with Net_Weight_oz <= 0:       {(data['Net_Weight_oz'] <= 0).sum()}")
print(f"Rows with Pack_Size <= 0:           {(data['Pack_Size'] <= 0).sum()}")
print(f"Rows with Shelf_Facings <= 0:       {(data['Shelf_Facings'] <= 0).sum()}")
print(f"Rows with Weekly_Revenue_USD <= 0:  {(data['Weekly_Revenue_USD'] <= 0).sum()}")

### **Exploratory Data Analysis — Univariate Numeric Distributions**

**Summary:** Histograms with kernel-density overlays were rendered for each numeric column to surface shape (skew), spread, and any visible outliers or multi-modality. This is the first look at where price, weight, and revenue concentrate.

**Observations:** _to be populated after execution._ Expected: `Weekly_Revenue_USD` is right-skewed (a few high-revenue item-store-weeks pull the upper tail), prices show stepwise concentration at common price points (e.g., \$0.99, \$1.99, \$2.99), and `Store_Sq_Ft` shows a discrete distribution corresponding to a small number of distinct store sizes.

In [ ]:
# ------------------------------
# UNIVARIATE EDA — NUMERIC DISTRIBUTIONS
# ------------------------------
# Plot a histogram with KDE overlay for each numeric column. Layout is
# 3 columns wide; rows expand as needed.

numeric_cols_for_eda = [
    "Net_Weight_oz",
    "Pack_Size",
    "List_Price_USD",
    "Promo_Price_USD",
    "Shelf_Facings",
    "Store_Sq_Ft",
    "Store_Open_Year",
    "Weekly_Units_Sold",
    "Weekly_Revenue_USD",
]

n_cols = 3                                                  # plot grid width
n_rows = (len(numeric_cols_for_eda) + n_cols - 1) // n_cols # rounded-up row count
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()                                       # flatten to index linearly

for i, col in enumerate(numeric_cols_for_eda):
    sns.histplot(data[col], kde=True, ax=axes[i], color="steelblue")
    axes[i].set_title(col)

# Hide any leftover empty subplots in the final row.
for j in range(len(numeric_cols_for_eda), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

### **Exploratory Data Analysis — Univariate Categorical Frequencies**

**Summary:** Value-count bar charts were rendered for each categorical column (excluding the high-cardinality `UPC` and `Item_Description`) to surface category balance, dominant levels, and any unexpected values. The high-cardinality columns get a `value_counts().head()` printout instead of a chart.

**Observations:** _to be populated after execution._

In [ ]:
# ------------------------------
# UNIVARIATE EDA — CATEGORICAL FREQUENCIES
# ------------------------------
# Bar-chart frequencies for low-cardinality categoricals; print head counts
# for the high-cardinality identifiers.

low_card_categoricals = [
    "Department",
    "Brand_Type",
    "Store_Number",
    "Store_Banner",
    "Store_Region",
]

high_card_categoricals = [
    "UPC",                                                  # 12-digit identifier; dropped before modeling
    "Item_Description",                                     # product names; kept as categorical
]

n_cols = 2
n_rows = (len(low_card_categoricals) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(low_card_categoricals):
    order = data[col].value_counts().index                  # bars sorted by frequency desc
    sns.countplot(data=data, y=col, order=order, ax=axes[i], color="steelblue")
    axes[i].set_title(col)

for j in range(len(low_card_categoricals), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

print()
print("=== High-cardinality categorical previews (top 10 by frequency) ===")
for col in high_card_categoricals:
    print(f"\n--- {col} (nunique = {data[col].nunique()}) ---")
    print(data[col].value_counts().head(10))

### **Exploratory Data Analysis — Bivariate Relationships**

**Summary:** Scatter plots of each numeric feature against the target `Weekly_Revenue_USD` were rendered to surface obvious linear / monotone / nonlinear relationships. This is the visual sanity check that the features carry signal before we commit to the model bake-off.

**Observations:** _to be populated after execution._ Expected: `List_Price_USD` and `Promo_Price_USD` should show a strong positive monotone relationship with revenue (higher price points generally correspond to higher per-row revenue). `Shelf_Facings` and `Store_Sq_Ft` should show a positive but noisier relationship. `Weekly_Units_Sold` will show the strongest relationship of all — this is the target-leakage signal that motivates dropping it.

In [ ]:
# ------------------------------
# BIVARIATE EDA — NUMERIC SCATTERS vs TARGET
# ------------------------------
# One scatter per numeric feature vs Weekly_Revenue_USD. Includes
# Weekly_Units_Sold so the leakage relationship is visually obvious.

bivariate_features = [
    "Net_Weight_oz",
    "Pack_Size",
    "List_Price_USD",
    "Promo_Price_USD",
    "Shelf_Facings",
    "Store_Sq_Ft",
    "Store_Open_Year",
    "Weekly_Units_Sold",                                    # included to show leakage; dropped before modeling
]

n_cols = 2
n_rows = (len(bivariate_features) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(bivariate_features):
    sns.scatterplot(data=data, x=col, y="Weekly_Revenue_USD", ax=axes[i], alpha=0.4, s=15)
    axes[i].set_title(f"{col} vs Weekly_Revenue_USD")

for j in range(len(bivariate_features), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

### **Correlation Matrix**

**Summary:** Pearson correlations among all numeric columns (including `Weekly_Units_Sold` and the target) were computed and rendered as a heatmap. The point of including `Weekly_Units_Sold` here is to make the target-leakage signal numerically explicit before the feature-engineering cell drops it.

**Observations:** _to be populated after execution._ Expected: `Weekly_Units_Sold` should show a near-1.0 correlation with `Weekly_Revenue_USD` (because revenue is approximately units × price). This is precisely the leakage that motivates dropping it.

In [ ]:
# ------------------------------
# CORRELATION MATRIX
# ------------------------------
# Pearson correlations across numeric columns, rendered as a heatmap.
# Kept Weekly_Units_Sold in this view to make the leakage explicit.

numeric_cols_for_corr = [
    "Net_Weight_oz",
    "Pack_Size",
    "List_Price_USD",
    "Promo_Price_USD",
    "Shelf_Facings",
    "Store_Sq_Ft",
    "Store_Open_Year",
    "Weekly_Units_Sold",                                    # included to expose the leakage signal
    "Weekly_Revenue_USD",
]

corr = data[numeric_cols_for_corr].corr(method="pearson")  # default Pearson correlation

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    square=True,
    cbar_kws={"shrink": 0.8},
)
plt.title("Pearson Correlation — Numeric Columns")
plt.tight_layout()
plt.show()

### **Revenue Aggregation by Segment**

**Summary:** Mean and total `Weekly_Revenue_USD` were computed for each level of the major categorical segments (`Department`, `Brand_Type`, `Store_Banner`, `Store_Region`). This surfaces which segments materially drive revenue and confirms that each categorical feature carries directional signal worth keeping in the model.

**Observations:** _to be populated after execution._

In [ ]:
# ------------------------------
# REVENUE AGGREGATION BY SEGMENT
# ------------------------------
# Group-by aggregation: mean and total Weekly_Revenue_USD per level of
# the four major categorical segments.

segment_cols = ["Department", "Brand_Type", "Store_Banner", "Store_Region"]

for col in segment_cols:
    print(f"\n=== Weekly_Revenue_USD by {col} ===")
    summary = (
        data.groupby(col)["Weekly_Revenue_USD"]
        .agg(["mean", "sum", "count"])
        .sort_values("sum", ascending=False)
        .rename(columns={"mean": "Mean_Revenue", "sum": "Total_Revenue", "count": "N_Rows"})
    )
    display(summary)

### **Target Leakage Callout — Why `Weekly_Units_Sold` is Dropped**

**Summary:** `Weekly_Units_Sold` is dropped from the feature set before any model is fit. It is not a usable input feature, despite being present in the source CSV — it is a target-leakage feature that trivializes the prediction problem.

The mechanical reason: revenue is approximately `units × price`. Both `List_Price_USD` and `Promo_Price_USD` are already in the feature set, so a model handed `Weekly_Units_Sold` as an input can recover the target almost exactly by multiplication. The correlation matrix above makes this numerically explicit (correlation with the target is near 1.0).

The business reason: a revenue forecaster is meant to predict revenue *before* the week has occurred, from product and store attributes that are known in advance — price, weight, pack size, shelf facings, store size, region. If the model required knowing units sold to predict revenue, the model would not be a forecaster. It would be a calculator that requires the answer as an input.

**Comparability note vs the prior generation.** The previous notebook in this engagement folder (`colab_authored.ipynb`, written against a 12-column retail CSV) **retained an analogous leakage feature** of similar predictive character, which produced reported R² and MAPE values that look superficially better than what this notebook is expected to achieve. Direct numeric comparison between the two notebooks' metrics is therefore not meaningful. This notebook's performance band (R² approximately 0.85-0.95, MAPE 8-15%) reflects honest out-of-sample prediction from product and store attributes alone.

**Observations:** The drop is implemented in the Feature Engineering cell below. The column is retained in the EDA cells above so the leakage signal is visible to the reader, then explicitly removed before the model pipeline sees the data.

### **Feature Engineering**

**Summary:** Four transformations were applied to prepare the dataset for modeling: (1) `Weekly_Units_Sold` was dropped (target leakage — see callout above). (2) `UPC` was dropped (12-digit unique identifier with cardinality so high that one-hot encoding would explode the feature matrix and overfit; `Item_Description`, `Department`, and `Brand_Type` carry the product-level signal). (3) `discount_pct` was derived as `(List_Price - Promo_Price) / List_Price`, capturing promotional depth as a single continuous feature. (4) `store_age` was derived from `Store_Open_Year` relative to a fixed reference year so the model sees a clean numeric tenure feature rather than a year value that drifts with calendar time. (5) `is_promo` was derived as a binary indicator (`Promo_Price < List_Price`) so the model can pick up the promotional-week effect directly. `Store_Open_Year` is dropped after `store_age` is derived since they encode the same information.

**Observations:** _to be populated after execution._ Expected: the cleaned feature set has the following columns plus the target — `Item_Description`, `Department`, `Brand_Type`, `Store_Number`, `Store_Banner`, `Store_Region`, `Net_Weight_oz`, `Pack_Size`, `List_Price_USD`, `Promo_Price_USD`, `Shelf_Facings`, `Store_Sq_Ft`, `discount_pct`, `store_age`, `is_promo`.

In [ ]:
# ------------------------------
# FEATURE ENGINEERING
# ------------------------------
# Drop target-leakage and high-cardinality identifier columns; derive
# discount_pct, store_age, and is_promo. Drop Store_Open_Year after
# store_age is computed.

REFERENCE_YEAR = 2025                                       # anchor year for store_age; pinned so the feature is reproducible

# Drop target leakage and the unusable identifier column.
data = data.drop(columns=["Weekly_Units_Sold", "UPC"])

# Derive discount_pct: relative promotional depth (0 = no discount, 1 = free).
data["discount_pct"] = (data["List_Price_USD"] - data["Promo_Price_USD"]) / data["List_Price_USD"]

# Derive store_age: years since the store opened, relative to REFERENCE_YEAR.
data["store_age"] = REFERENCE_YEAR - data["Store_Open_Year"]
data = data.drop(columns=["Store_Open_Year"])              # store_age now encodes the same information

# Derive is_promo: 1 if Promo_Price is strictly less than List_Price, else 0.
data["is_promo"] = (data["Promo_Price_USD"] < data["List_Price_USD"]).astype(int)

print(f"Post-engineering shape: {data.shape}")
print(f"Columns: {list(data.columns)}")
data.head()

### **Train / Test Split**

**Summary:** The engineered dataset was split into 70% training and 30% holdout test, with a pinned random state for reproducibility. The test split is set aside untouched until final evaluation — no model selection or hyperparameter tuning sees the test data.

**Observations:** _to be populated after execution._ Expected: roughly 6,216 training rows and 2,664 test rows.

In [ ]:
# ------------------------------
# TRAIN / TEST SPLIT
# ------------------------------
# Separate the target from the features, then split 70/30 with a pinned
# random state. The test split is untouched until final evaluation.

TARGET_COL = "Weekly_Revenue_USD"

X = data.drop(columns=[TARGET_COL])                         # feature DataFrame
y = data[TARGET_COL]                                        # target Series

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=HOLDOUT_TEST_SIZE,
    random_state=RANDOM_STATE,
)

print(f"Train shape: X={X_train.shape}, y={y_train.shape}")
print(f"Test shape:  X={X_test.shape}, y={y_test.shape}")

### **Preprocessing Pipeline**

**Summary:** A `ColumnTransformer` was defined that applies median imputation to numeric columns and most-frequent imputation followed by one-hot encoding to categorical columns. The transformer is wrapped inside each model's `Pipeline` so cross-validation folds are preprocessed using only the fold's training portion — preventing the kind of leakage that happens when preprocessing is done on the full dataset before splitting.

**Observations:** Pipeline-based preprocessing is the standard pattern from `notebook-style-guide.md` v2.1. The fitted preprocessor is also serialized separately in the Model Serialization section so a downstream frontend can transform user inputs without re-fitting.

In [ ]:
# ------------------------------
# PREPROCESSING PIPELINE
# ------------------------------
# Define numeric and categorical column lists, then assemble a
# ColumnTransformer that runs each column group through its own
# preprocessing pipeline.

numeric_features = [
    "Net_Weight_oz",
    "Pack_Size",
    "List_Price_USD",
    "Promo_Price_USD",
    "Shelf_Facings",
    "Store_Sq_Ft",
    "discount_pct",
    "store_age",
    "is_promo",                                             # binary 0/1, treated as numeric pass-through
]

categorical_features = [
    "Item_Description",
    "Department",
    "Brand_Type",
    "Store_Number",
    "Store_Banner",
    "Store_Region",
]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),          # fill numeric NaNs with column median (defensive even if data is clean)
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),   # fill categorical NaNs with column mode
    ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),  # OHE; unknown categories at inference become all-zero
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop",                                       # any column not listed above is dropped
)

print("Preprocessor defined.")
print(f"  Numeric features ({len(numeric_features)}):     {numeric_features}")
print(f"  Categorical features ({len(categorical_features)}): {categorical_features}")

### **Model Definitions and Hyperparameter Grids**

**Summary:** Six regression estimators were defined for the bake-off (Decision Tree, Bagging, Random Forest, Gradient Boosting, XGBoost, CatBoost). Each gets a model-specific hyperparameter grid sized so the bake-off completes in 8-15 minutes on standard Colab CPU runtime. All estimators are pinned to the shared `RANDOM_STATE`; individual estimators run single-threaded so `GridSearchCV`'s outer parallelism controls overall concurrency without oversubscribing CPUs.

**Observations:** The grids are deliberately modest — depth in {3, 5, 7, None}, n_estimators in {100, 200}, learning rate in {0.05, 0.1}. The point of the bake-off is to compare model *families* on this dataset, not to push any single family to its limit. Tighter tuning is a separate task.

In [ ]:
# ------------------------------
# MODEL DEFINITIONS AND HYPERPARAMETER GRIDS
# ------------------------------
# Six estimators with their tuning grids. Random states pinned to
# RANDOM_STATE. Single-thread (n_jobs=1, thread_count=1) inside individual
# estimators so GridSearchCV's outer parallelism controls concurrency.

MODEL_DISPLAY_LABELS = {                                    # human-readable labels used in tables and the model registry
    "Decision Tree":      "Decision Tree",
    "Bagging":            "Bagging",
    "Random Forest":      "Random Forest",
    "Gradient Boosting":  "Gradient Boosting",
    "XGBoost":            "XGBoost",
    "CatBoost":           "CatBoost",
}

MODEL_SLUGS = {                                             # filesystem-safe slugs used in serialized model filenames
    "Decision Tree":     "decision-tree",
    "Bagging":           "bagging",
    "Random Forest":     "random-forest",
    "Gradient Boosting": "gradient-boosting",
    "XGBoost":           "xgboost",
    "CatBoost":          "catboost",
}

models = {
    "Decision Tree":     DecisionTreeRegressor(random_state=RANDOM_STATE),
    "Bagging":           BaggingRegressor(random_state=RANDOM_STATE, n_jobs=1),
    "Random Forest":     RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    "XGBoost":           XGBRegressor(
        random_state=RANDOM_STATE,
        objective="reg:squarederror",
        verbosity=0,
        n_jobs=1,
    ),
    "CatBoost":          CatBoostRegressor(
        random_state=RANDOM_STATE,
        verbose=0,
        allow_writing_files=False,
        thread_count=1,
    ),
}

param_grids = {
    "Decision Tree": {
        "regressor__max_depth":        [3, 5, 7, None],
        "regressor__min_samples_leaf": [1, 5, 10],
    },
    "Bagging": {
        "regressor__n_estimators": [50, 100],
        "regressor__max_samples":  [0.7, 1.0],
    },
    "Random Forest": {
        "regressor__n_estimators": [100],
        "regressor__max_depth":    [10, None],
        "regressor__max_features": ["sqrt", None],
    },
    "Gradient Boosting": {
        "regressor__n_estimators":  [100, 200],
        "regressor__learning_rate": [0.05, 0.1],
        "regressor__max_depth":     [3, 5],
    },
    "XGBoost": {
        "regressor__n_estimators":     [100, 200],
        "regressor__learning_rate":    [0.05, 0.1],
        "regressor__max_depth":        [3, 5],
        "regressor__subsample":        [0.8],
        "regressor__colsample_bytree": [0.8],
    },
    "CatBoost": {
        "regressor__iterations":    [300, 500],
        "regressor__learning_rate": [0.05, 0.1],
        "regressor__depth":         [4, 6, 8],
    },
}

print(f"Defined {len(models)} models for bake-off: {list(models.keys())}")

### **Model Evaluation Utilities**

**Summary:** Two helper functions were defined: `get_metrics` returns RMSE, MAE, R², adjusted R², and MAPE for a (y_true, y_pred) pair labeled by subset name; `evaluate_model` runs the full per-model evaluation flow (base fit, `GridSearchCV` on the training split, repeated K-fold cross-validation, tuned-model train/test metrics). The structured return dict per model is what the bake-off loop populates.

**Observations:** Repeated K-fold (5 splits × 2 repeats = 10 folds) is the primary ranking signal because it produces a more stable estimate of out-of-sample RMSE than a single 70/30 split. The holdout test set is for final confirmation, not for selection.

In [ ]:
# ------------------------------
# MODEL EVALUATION UTILITIES
# ------------------------------
# Helper functions for consistent metric computation and per-model
# evaluation across the six-model bake-off.

def get_metrics(y_true, y_pred, subset, feature_count):
    """Return RMSE / MAE / R-squared / Adj R-squared / MAPE for a prediction set, keyed by subset name."""
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    denominator = max(len(y_true) - feature_count - 1, 1)               # protect against degenerate denominators
    adj_r2 = 1 - (1 - r2) * (len(y_true) - 1) / denominator
    mape = float(mean_absolute_percentage_error(y_true, y_pred))
    return {
        f"{subset}_RMSE":   rmse,
        f"{subset}_MAE":    mae,
        f"{subset}_R2":     r2,
        f"{subset}_Adj_R2": adj_r2,
        f"{subset}_MAPE":   mape,
    }


def evaluate_model(model_name, model, param_grid, X_train, y_train, X_test, y_test):
    """Full per-model evaluation: base fit, GridSearchCV, repeated K-fold, holdout test."""
    print(f"Processing {model_name}...")

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),                                  # shared ColumnTransformer
        ("regressor",    model),
    ])
    feature_count = X_train.shape[1]

    # 1) Base fit and metrics for context (untuned reference point).
    pipeline.fit(X_train, y_train)
    base_metrics_train = get_metrics(y_train, pipeline.predict(X_train), "Train", feature_count)
    base_metrics_test  = get_metrics(y_test,  pipeline.predict(X_test),  "Test",  feature_count)

    # 2) Hyperparameter tuning on the training split only.
    search = GridSearchCV(
        pipeline,
        param_grid,
        cv=SEARCH_CV,
        n_jobs=-1,                                                        # outer parallelism across the CV grid
        scoring="neg_root_mean_squared_error",
    )
    search.fit(X_train, y_train)
    best_model = search.best_estimator_

    # 3) Repeated K-fold validation on the training split for stable ranking.
    validation_scores = cross_validate(
        best_model,
        X_train, y_train,
        cv=REPEATED_VALIDATION_CV,
        scoring={
            "rmse": "neg_root_mean_squared_error",
            "mae":  "neg_mean_absolute_error",
            "r2":   "r2",
            "mape": "neg_mean_absolute_percentage_error",
        },
        n_jobs=-1,
        return_train_score=False,
    )
    validation_metrics = {
        "CV_RMSE_Mean": float(-validation_scores["test_rmse"].mean()),
        "CV_RMSE_Std":  float(validation_scores["test_rmse"].std()),
        "CV_MAE_Mean":  float(-validation_scores["test_mae"].mean()),
        "CV_R2_Mean":   float(validation_scores["test_r2"].mean()),
        "CV_MAPE_Mean": float(-validation_scores["test_mape"].mean()),
    }

    # 4) Tuned-model holdout metrics.
    tuned_train_metrics = get_metrics(y_train, best_model.predict(X_train), "Train", feature_count)
    tuned_test_metrics  = get_metrics(y_test,  best_model.predict(X_test),  "Test",  feature_count)

    return {
        "Base":       {**base_metrics_train, **base_metrics_test},
        "Tuned":      {
            **tuned_train_metrics,
            **tuned_test_metrics,
            "Best Params":      search.best_params_,
            "Fitted Estimator": best_model,
        },
        "Validation": validation_metrics,
    }

### **Model Training Loop**

**Summary:** All six models were run through `evaluate_model` and their structured results collected into a `results` dict keyed by model name. Wall-clock time depends on the Colab runtime's CPU count; the slowest single model is typically CatBoost.

**Observations:** _to be populated after execution._ Expected: total bake-off wall time of 8-15 minutes on standard Colab CPU runtime.

In [ ]:
# ------------------------------
# MODEL TRAINING LOOP
# ------------------------------
# Run the bake-off across all six models. The results dict carries the
# fitted tuned estimator for each model so we can serialize all six later.

results = {}

for name, model in models.items():
    results[name] = evaluate_model(
        name,
        model,
        param_grids[name],
        X_train, y_train,
        X_test,  y_test,
    )

print("\nAll six models trained and tuned.")

### **Model Comparison — Performance Aggregation**

**Summary:** The per-model `results` dict was flattened into two DataFrames: `validation_perf_df` (repeated K-fold metrics) and `test_perf_df` (tuned-model holdout metrics). Both are sorted to make the ranking immediately readable.

**Observations:** _to be populated after execution._

In [ ]:
# ------------------------------
# PERFORMANCE AGGREGATION
# ------------------------------
# Flatten the nested results dict into two DataFrames: validation
# (repeated K-fold) and holdout test. Sort each by primary ranking metric.

validation_rows = []
test_rows = []

for name, r in results.items():
    val = r["Validation"]
    tuned = r["Tuned"]
    validation_rows.append({
        "Model":     name,
        "RMSE":      val["CV_RMSE_Mean"],
        "RMSE Std":  val["CV_RMSE_Std"],
        "MAE":       val["CV_MAE_Mean"],
        "MAPE":      val["CV_MAPE_Mean"],
        "R-squared": val["CV_R2_Mean"],
    })
    test_rows.append({
        "Model":          name,
        "RMSE":           tuned["Test_RMSE"],
        "MAE":            tuned["Test_MAE"],
        "MAPE":           tuned["Test_MAPE"],
        "R-squared":      tuned["Test_R2"],
        "Adj. R-squared": tuned["Test_Adj_R2"],
    })

validation_perf_df = pd.DataFrame(validation_rows).set_index("Model").sort_values("RMSE")
test_perf_df = pd.DataFrame(test_rows).set_index("Model").sort_values("RMSE")

print("=== Validation Performance (Repeated K-Fold) ===")
display(validation_perf_df)

print("\n=== Holdout Test Performance (Tuned Model) ===")
display(test_perf_df)

### **Primary and Secondary Model Selection**

**Summary:** The two model frames were joined and the combined frame sorted by validation RMSE (primary ranking signal) with MAE / holdout RMSE / R² as tie-breakers. The top-ranked model becomes the primary; the second-ranked becomes the secondary. Both are deployed-eligible; the frontend's admin dropdown lets the user flip between any of the six serialized candidates.

**Observations:** _to be populated after execution._

In [ ]:
# ------------------------------
# BEST MODEL SELECTION
# ------------------------------
# Rank by repeated-validation RMSE; confirm with holdout test metrics.
# Top two models become primary + secondary for serialization metadata.

model_selection_df = (
    validation_perf_df.rename(columns={
        "RMSE":      "Validation_RMSE",
        "RMSE Std":  "Validation_RMSE_Std",
        "MAE":       "Validation_MAE",
        "MAPE":      "Validation_MAPE",
        "R-squared": "Validation_R2",
    })
    .join(
        test_perf_df.rename(columns={
            "RMSE":           "Holdout_RMSE",
            "MAE":            "Holdout_MAE",
            "MAPE":           "Holdout_MAPE",
            "R-squared":      "Holdout_R2",
            "Adj. R-squared": "Holdout_Adj_R2",
        }),
        how="left",
    )
    .sort_values(
        by=["Validation_RMSE", "Validation_MAE", "Holdout_RMSE", "Validation_R2"],
        ascending=[True, True, True, False],
    )
)

top_two_models = model_selection_df.index[:2].tolist()
primary_model_name = top_two_models[0]
secondary_model_name = top_two_models[1]

PRIMARY_MODEL_LABEL = MODEL_DISPLAY_LABELS[primary_model_name]
SECONDARY_MODEL_LABEL = MODEL_DISPLAY_LABELS[secondary_model_name]

print("=== Final Model Ranking ===")
display(model_selection_df)

print(f"\nPrimary model:   {primary_model_name} -> displayed as '{PRIMARY_MODEL_LABEL}'")
print(f"Secondary model: {secondary_model_name} -> displayed as '{SECONDARY_MODEL_LABEL}'")

print(f"\nPrimary best params:   {results[primary_model_name]['Tuned']['Best Params']}")
print(f"Secondary best params: {results[secondary_model_name]['Tuned']['Best Params']}")

### **Test Set Evaluation — Primary Model**

**Summary:** The primary model's predictions on the held-out test set were used to compute final RMSE, MAE, R², adjusted R², and MAPE. A predicted-vs-actual scatter plot was rendered alongside the metrics so the residual structure is visible (heteroscedasticity, systematic bias at high or low revenue values).

**Observations:** _to be populated after execution._

In [ ]:
# ------------------------------
# TEST SET EVALUATION — PRIMARY MODEL
# ------------------------------
# Pull the primary model's tuned holdout metrics into named scalars so
# subsequent cells (tier assignment, business alignment, executive
# summary) can reference them by name.

primary_tuned = results[primary_model_name]["Tuned"]

test_rmse   = primary_tuned["Test_RMSE"]
test_mae    = primary_tuned["Test_MAE"]
test_r2     = primary_tuned["Test_R2"]
test_adj_r2 = primary_tuned["Test_Adj_R2"]
test_mape   = primary_tuned["Test_MAPE"]

test_mean_actual = float(y_test.mean())                     # for business-alignment framing

print(f"=== Primary model: {primary_model_name} — Test Set Metrics ===")
print(f"  Test RMSE:    ${test_rmse:,.2f}")
print(f"  Test MAE:     ${test_mae:,.2f}")
print(f"  Test R²:      {test_r2:.4f}")
print(f"  Test Adj R²:  {test_adj_r2:.4f}")
print(f"  Test MAPE:    {test_mape*100:.2f}%")
print(f"  Mean actual:  ${test_mean_actual:,.2f}")

In [ ]:
# ------------------------------
# PREDICTED VS ACTUAL VISUALIZATION — PRIMARY MODEL
# ------------------------------
# Scatter of predicted vs actual Weekly_Revenue_USD on the test set,
# with a 45-degree reference line. Tight clustering around the line is
# the visual confirmation of low residual error.

primary_model = results[primary_model_name]["Tuned"]["Fitted Estimator"]
y_pred_test = primary_model.predict(X_test)

plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred_test, alpha=0.4, s=15)

min_val = min(y_test.min(), y_pred_test.min())
max_val = max(y_test.max(), y_pred_test.max())
plt.plot([min_val, max_val], [min_val, max_val], color="red", linestyle="--", label="Perfect prediction (y = x)")

plt.xlabel("Actual Weekly_Revenue_USD")
plt.ylabel("Predicted Weekly_Revenue_USD")
plt.title(f"Predicted vs Actual — {primary_model_name} (Test Set)")
plt.legend()
plt.tight_layout()
plt.show()

### **Business Alignment and Tier Assignment**

**Summary:** The catalog's v1.0 quality-tier thresholds were applied to the measured test metrics, then each business opportunity from the Executive Summary was mapped to the metric that supports it, with plain-English framing per `system-design.md` §12.7. The v1.0 thresholds are kept verbatim — the operator's stated R² 0.85-0.95 / MAPE 8-15% target band on this dataset places a "good" outcome in Tier 2 (Shippable — Strong) rather than Tier 1; this is expected and reflects the absence of the `Weekly_Units_Sold` leakage feature that the prior generation of this engagement retained.

**Observations:** _to be populated after execution._

In [ ]:
# ------------------------------
# BUSINESS ALIGNMENT AND TIER ASSIGNMENT
# ------------------------------
# Apply the v1.0 catalog tier thresholds to the measured test metrics,
# then frame each business opportunity in plain English using the
# measured numbers.

QUALITY_TIERS = {                                           # v1.0 thresholds, kept verbatim
    "Tier 1: Shippable — Exceptional":  {"min_r2": 0.92, "max_mape": 0.08},
    "Tier 2: Shippable — Strong":       {"min_r2": 0.85, "max_mape": 0.12},
    "Tier 3: Shippable — Below target": {"min_r2": 0.70, "max_mape": 0.20},
    # Tier 4 = fallthrough below all of the above
}

assigned_tier = "Tier 4: Not Shippable — We do not recommend using this model"
for tier, criteria in QUALITY_TIERS.items():
    if test_r2 >= criteria["min_r2"] and test_mape <= criteria["max_mape"]:
        assigned_tier = tier
        break

print("=== Tier Assignment ===")
print(f"  Tier:    {assigned_tier}")
print(f"  Receipt: R² = {test_r2:.4f}, MAPE = {test_mape*100:.2f}%")
print()
print("=== Business Alignment — mapping measured metrics back to opportunities ===")
print()
print("A. Forecast weekly revenue per item-store combination consistently:")
print(f"   MAE ${test_mae:,.2f} on a mean actual of ${test_mean_actual:,.2f} means forecasts are")
print(f"   typically within {test_mape*100:.1f}% of actual weekly revenue — the kind of band a")
print(f"   category manager already absorbs in week-to-week planning.")
print()
print("B. Plan shelf allocation and promotional pricing with measured confidence intervals:")
print(f"   MAPE {test_mape*100:.1f}% gives planners a measured confidence band: forecasts are")
print(f"   typically within roughly {test_mape*100:.0f}% of actual on any given item-store row.")
print()
print("C. Identify mispriced, misallocated, or under-promoted inventory:")
two_x_mape = test_mape * 100 * 2
print(f"   An item-store row's actual sales more than {two_x_mape:.0f}% off the model's forecast is")
print(f"   a meaningful flag worth investigating; the model's {test_mape*100:.1f}% MAPE is the noise")
print(f"   floor below which discrepancies are likely normal week-to-week variation rather than")
print(f"   actionable mispricing or misallocation signal.")

### **Model Serialization — Primary and Secondary**

**Summary:** The primary and secondary fitted estimators were serialized to `joblib` files inside the engagement's `models/` directory. These two are the headline outputs of the bake-off and what a default deployment would load. The next cell extends serialization to all six candidates plus the preprocessor.

**Observations:** Filename convention is `{model_slug}__hill-country-grocer.joblib`. The `models/` directory is created if it does not already exist.

In [ ]:
# ------------------------------
# MODEL SERIALIZATION — PRIMARY AND SECONDARY
# ------------------------------
# Write the primary and secondary fitted estimators to joblib files in
# the engagement's models/ directory. Filename pattern:
# {model_slug}__hill-country-grocer.joblib

MODELS_DIR = "models"                                       # relative to notebook cwd in Colab
os.makedirs(MODELS_DIR, exist_ok=True)

def serialize_model(model_name, fitted_estimator):
    """Write a fitted estimator to disk using the canonical filename pattern."""
    slug = MODEL_SLUGS[model_name]
    out_path = os.path.join(MODELS_DIR, f"{slug}__hill-country-grocer.joblib")
    joblib.dump(fitted_estimator, out_path)
    print(f"  Wrote {out_path}")
    return out_path

print("Serializing primary and secondary models...")
serialize_model(primary_model_name,   results[primary_model_name]["Tuned"]["Fitted Estimator"])
serialize_model(secondary_model_name, results[secondary_model_name]["Tuned"]["Fitted Estimator"])

### **Model Serialization — All Candidates and Preprocessor**

**Summary:** All six trained candidates plus the fitted preprocessor were serialized to disk, and a `model_registry.json` was written alongside them. The registry records each model's slug, display name, test R², test MAPE, and whether it was selected as primary at training time. This supports a downstream frontend that exposes a model-choice dropdown across the full bake-off (not just the primary), so end users can compare predictions across model families. The fitted preprocessor is serialized separately because the frontend needs to transform user inputs identically to how the training pipeline transformed them.

**Observations:** Per the engagement prompt, this notebook does NOT push the serialized artifacts to GitHub. They are written to local disk during Colab execution; a separate commit task handles uploading them to reference-data.

In [ ]:
# ------------------------------
# MODEL SERIALIZATION — ALL CANDIDATES AND PREPROCESSOR
# ------------------------------
# Persist every trained candidate plus the fitted preprocessor, then
# write a model_registry.json describing them. Test R² and test MAPE
# come from the per-model results dict.

print("Serializing all six candidates...")
for name in models.keys():
    fitted = results[name]["Tuned"]["Fitted Estimator"]
    serialize_model(name, fitted)

# The preprocessor lives inside each model's Pipeline (named "preprocessor").
# Pull it from the primary estimator and serialize it separately so the
# frontend can transform user inputs without re-fitting.
fitted_preprocessor = (
    results[primary_model_name]["Tuned"]["Fitted Estimator"]
    .named_steps["preprocessor"]
)
preprocessor_path = os.path.join(MODELS_DIR, "preprocessor__hill-country-grocer.joblib")
joblib.dump(fitted_preprocessor, preprocessor_path)
print(f"  Wrote {preprocessor_path}")

# Build the registry and write it as JSON.
registry = {
    "engagement_slug":      "hill-country-grocer__reg__ensemble-exp",
    "primary_model_slug":   MODEL_SLUGS[primary_model_name],
    "secondary_model_slug": MODEL_SLUGS[secondary_model_name],
    "preprocessor_file":    "preprocessor__hill-country-grocer.joblib",
    "models": [],
}

for name in models.keys():
    tuned = results[name]["Tuned"]
    registry["models"].append({
        "slug":         MODEL_SLUGS[name],
        "display_name": MODEL_DISPLAY_LABELS[name],
        "file":         f"{MODEL_SLUGS[name]}__hill-country-grocer.joblib",
        "test_r2":      float(tuned["Test_R2"]),
        "test_mape":    float(tuned["Test_MAPE"]),
        "is_primary":   (name == primary_model_name),
        "is_secondary": (name == secondary_model_name),
        "best_params":  {str(k): v for k, v in tuned["Best Params"].items()},
    })

registry_path = os.path.join(MODELS_DIR, "model_registry.json")
with open(registry_path, "w") as f:
    json.dump(registry, f, indent=2, default=str)
print(f"  Wrote {registry_path}")

print("\nSerialization complete. Six candidates + preprocessor + registry on disk.")

---

## **Expanded Executive Summary**

### **TL;DR**

A six-model regression bake-off was trained on 8,880 weekly item-store sales records from a synthetic Hill Country Grocer (Texas regional grocery chain) panel. Models were ranked by repeated K-fold cross-validation RMSE and confirmed on a 30% held-out test set. The target is `Weekly_Revenue_USD` — predicted from product and store attributes alone, with `Weekly_Units_Sold` deliberately dropped from the feature set to avoid target leakage. The primary model's test-set MAE, MAPE, R², and tier assignment are reported in the Test Set Evaluation and Business Alignment sections above (filled in after Colab execution). All six trained candidates are serialized to disk to support a downstream frontend that exposes a model-choice dropdown across the full bake-off; the primary model determined by validation RMSE is flagged in `model_registry.json` but the frontend ultimately exposes choice across all six.

### **Full Summary**

#### **Objective**

Build a deployable weekly-revenue forecaster for a multi-banner Texas regional grocery chain that gives category managers and inventory planners a defensible reference forecast for any item-store combination in their catalog, from product and store attributes that are known *before* the week begins (price, weight, pack size, shelf facings, store banner, region, age) — never from inputs that would only be known after the fact, like units sold.

#### **Data and Preparation**

The dataset (8,880 rows by 16 columns) contained product attributes (UPC, item description, department, brand type, net weight, pack size, list price, promo price, shelf facings), store attributes (number, banner, region, square footage, open year), `Weekly_Units_Sold`, and the target `Weekly_Revenue_USD`. Missing values, duplicates, and obviously implausible numerics were checked in the Data Quality Checks cell. Cleaning was minimal where the synthetic dataset was already well-formed.

Feature engineering:

- **Dropped `Weekly_Units_Sold`** — target-leakage feature (units × price approximately equals revenue). This is the single most important modeling choice in the notebook. The prior generation of this engagement retained an analogous leakage feature, which means its reported R² and MAPE values are not directly comparable to this notebook's measured numbers.

- **Dropped `UPC`** — 12-digit unique product identifier with cardinality high enough that one-hot encoding would explode the feature matrix and overfit. Product-level signal is carried by `Item_Description`, `Department`, and `Brand_Type` instead.

- **Derived `discount_pct`** — `(List_Price - Promo_Price) / List_Price`. Captures promotional depth as a continuous feature.

- **Derived `store_age`** — `2025 - Store_Open_Year`. Anchored to a fixed reference year so the feature is reproducible.

- **Derived `is_promo`** — binary indicator (`Promo_Price < List_Price`). Captures the promotional-week effect directly.

- **Dropped `Store_Open_Year`** — redundant with `store_age`.

Train/test split: 70/30 with `random_state=1`. The test set was untouched until final evaluation.

#### **Iterative Development**

Six model architectures: `Decision Tree`, `Bagging`, `Random Forest`, `Gradient Boosting`, `XGBoost`, `CatBoost`. Each was wrapped in an sklearn `Pipeline` with a `ColumnTransformer` preprocessor (median imputation for numeric features, most-frequent imputation followed by one-hot encoding for categorical features). Each was tuned via 5-fold `GridSearchCV` with model-specific hyperparameter grids (depth, learning rate, n_estimators, max_features, subsample), then validated via repeated K-fold (5 splits × 2 repeats = 10 folds total) for stable model ranking.

#### **Model Selection**

Ranking metric: repeated-validation RMSE (per `system-design.md` §6.1). Confirmation metric: holdout test RMSE. The top-two ranked models become primary and secondary. Tree-based gradient-boosting methods (CatBoost, XGBoost, Gradient Boosting) typically lead on tabular datasets of this shape; the Bagging meta-estimator and Random Forest are strong runners-up. The Decision Tree provides a low-complexity baseline for context.

#### **Tier Assignment**

The v1.0 catalog thresholds were applied verbatim (Tier 1: R² ≥ 0.92 ∧ MAPE ≤ 8%; Tier 2: R² ≥ 0.85 ∧ MAPE ≤ 12%; Tier 3: R² ≥ 0.70 ∧ MAPE ≤ 20%; Tier 4 otherwise). The operator's stated target band on this dataset (R² 0.85-0.95, MAPE 8-15%) is consistent with a Tier 2 (Shippable — Strong) landing; clearing Tier 1 with margin would require a feature stronger than the ones available without `Weekly_Units_Sold`.

#### **Deployment Readiness**

All six trained candidates are serialized via `joblib`, alongside the fitted `ColumnTransformer` preprocessor and a `model_registry.json` that records each model's slug, display name, test metrics, and whether it was primary at training time. The registry is the contract between this notebook and the downstream frontend: the frontend reads it to populate an admin model-choice dropdown so end users can compare predictions across the full bake-off, not just the primary.

This notebook does not push the serialized artifacts to GitHub. That happens in a follow-up task after the operator confirms the Colab-executed metrics.

#### **Limitations and Honest Framing**

- The dataset is synthetic. Real-world Hill Country Grocer data may exhibit seasonality, promotional cadence, holiday effects, and competitor-pricing signal that this synthetic panel does not encode. Deploying against real data would require recalibration.

- The model is a snapshot trained on a fixed historical dataset with no temporal validation (no walk-forward CV, no holdout-by-week). Real deployment would require periodic retraining and time-aware validation — see `system-design.md` §14 (Catalog Lifecycle: Retraining and Re-validation).

- MAPE is a chain-level average across all item-store-week rows. Specific item-store rows can be off by more or less than that. The predicted-vs-actual scatter in the Test Set Evaluation section shows the spread.

- Predictions for item-store combinations whose product description, department, brand type, store banner, or store region was not present in the training data fall outside the trained distribution; the one-hot encoder treats unknown categories as all-zero rows, which is a defensible fallback but produces less reliable predictions than for in-distribution rows.

---

*End of notebook.*